#### tensorfolow2.10以下，tf2.10以上对于优化器的调用方式不一样，见cell中详细说明

In [1]:
from tensorflow.keras.layers import Input, Dense, Reshape, Flatten, BatchNormalization, Activation, ZeroPadding2D,LeakyReLU

from tensorflow.keras.models import Sequential, Model

#from tensorflow.keras.optimizers import Adam   #tensorfolow2.10以下使用

from tensorflow.keras.optimizers.legacy import Adam   #tensorfolow2.10及以上使用

from tensorflow.keras.datasets import mnist

import matplotlib.pyplot as plt

import os
import numpy as np

In [2]:
#print(tf.__version__)

In [3]:
#在线调用mnist数据集，train函数中有直接调用，此处省略，只为了说明调用方法而设置。
#(X_train, _), (_, _) = mnist.load_data()
#X_train.shape

In [4]:
# --------------------------------- #
#   行28，列28，也就是mnist的中每个图片的大小
# --------------------------------- #
img_rows = 28
img_cols = 28
channels = 1
# 28,28,1
img_shape = (img_rows, img_cols, channels)
latent_dim = 100#随机噪声的长度。

In [5]:
optimizer_adam = Adam(0.0002, 0.5)

In [6]:
def build_generator():
    # --------------------------------- #
    #   生成器，输入一串随机数字
    # --------------------------------- #
    model = Sequential()

    model.add(Dense(256, input_dim=latent_dim))#输入一个100维的噪声
    model.add(LeakyReLU(alpha=0.2))
    model.add(BatchNormalization(momentum=0.8))

    model.add(Dense(512))
    model.add(LeakyReLU(alpha=0.2))
    model.add(BatchNormalization(momentum=0.8))

    model.add(Dense(1024))
    model.add(LeakyReLU(alpha=0.2))
    model.add(BatchNormalization(momentum=0.8))

    model.add(Dense(np.prod(img_shape), activation='tanh'))
    # np.prod计算所有元素的乘积，保证生成输出的维度与图片维度一致
    model.add(Reshape(img_shape))#model最后一层是把上一层的输出维度，重塑成图像尺寸28*28*1

    noise = Input(shape=(latent_dim,))
    img = model(noise)

    return Model(noise, img)#返回一个模型，输入是100维度的噪声和噪声经过model以后得到的图像。
generator = build_generator()

In [7]:
def build_discriminator():
    # ----------------------------------- #
    #   评价器，对输入进来的图片进行评价
    # ----------------------------------- #
    model = Sequential()
    # 输入一张图片
    model.add(Flatten(input_shape=img_shape))
    model.add(Dense(512))
    model.add(LeakyReLU(alpha=0.2))
    model.add(Dense(256))
    model.add(LeakyReLU(alpha=0.2))
    # 判断真伪，模型最后一层输出一个值
    model.add(Dense(1, activation='sigmoid'))

    img = Input(shape=img_shape)
    validity = model(img)
    discriminator = Model(img, validity)
     # 判别器
    #discriminator = D
    discriminator.compile(loss='binary_crossentropy',
                               optimizer=optimizer_adam,
                               metrics=['accuracy'])
    return discriminator
discriminator = build_discriminator()

In [8]:
def GAN():
 
    # adam优化器
    #optimizer_adam = Adam(0.0002, 0.5)
    # 判别器
    #discriminator = D
    #discriminator.compile(loss='binary_crossentropy',
                              # optimizer=optimizer_adam,
                              # metrics=['accuracy'])
    # 生成器
    #generator = G

    gan_input = Input(shape=(latent_dim,))
    img = generator(gan_input)
    # 在训练generate的时候不训练discriminator
    discriminator.trainable = False
    # 对生成的假图片进行预测
    validity = discriminator(img)
    combined = Model(gan_input, validity)
    combined.compile(loss='binary_crossentropy', optimizer=optimizer_adam, metrics=['accuracy'])
    return combined#返回Gan网络模型，其中鉴别器不可训练。
combined = GAN()

In [9]:
def sample_images(epoch):

    r, c = 5, 5
    noise = np.random.normal(0, 1, (r * c, latent_dim))#产生随机噪声，r*c行，100列
    gen_imgs = generator.predict(noise)#使用当前训练阶段的生成器对噪声进行预测，输出图片。r*c张28*28*1的图片
    

    gen_imgs = 0.5 * gen_imgs + 0.5

    fig, axs = plt.subplots(r, c)
    cnt = 0
    for i in range(r):
        for j in range(c):
            #print("----")
            axs[i, j].imshow(gen_imgs[cnt, :, :, 0], cmap='gray')
            axs[i, j].axis('off')
            #plt.imshow(gen_imgs[cnt, :, :, 0], cmap='gray')
            cnt += 1
    fig.savefig("gan_mnist/%d.png" % epoch)
    plt.close()

In [10]:
#(X_train, _), (_, _) = mnist.load_data()

In [11]:
#X_train.max()

In [12]:
#X_train.shape[0]

####  fit_genrator和fit，在指定batch=8的时候，他们模型是每8个数据对自己进行训练，训练完文件夹下所有数据集，fit_genrator在设定参数上面则是更为严格，与fit效果一样。train_on_batch则是使用8个数据集进行训练，并不能训练完所有数据集，如果想训练完所有数据集，则需要增加for来便利文件夹中所有数据。

In [23]:
def train(epochs, batch_size=128, sample_interval=50):
    # 获得数据
    (X_train, _), (_, _) = mnist.load_data()

    # 进行数据归一化到[-1, 1]
    X_train = X_train / 127.5 - 1.
    X_train = np.expand_dims(X_train, axis=3)

    # 创建标签
    valid = np.ones((batch_size, 1))#256行，值为1
    fake = np.zeros((batch_size, 1))#256行，值为0

    for epoch in range(epochs):#每个epoch

        # --------------------------- #
        #   随机选取batch_size个图片
        #   对discriminator进行训练
        #   训练鉴别器
        # --------------------------- #
        idx = np.random.randint(0, X_train.shape[0], batch_size)#随机产生0-60000范围中的任意256个整数。
        imgs = X_train[idx]#在训练集中，随机提取256个图像数据出来。
        #每个epoch都随机产生（256，100）的值在0-1中间的随机噪声。
        noise = np.random.normal(0, 1, (batch_size, latent_dim))
        #随机的噪声送入当前epoch后的generator开始预测结果。输出产生的图像，有256张。
        gen_imgs = generator.predict(noise)
        # train_on_batch返回loss(若模型编译时赋予metrics=['acc']则还返回acc)
        #使用随机抽取的256张真实图片，训练鉴别器，让他结果逼近valid（1），并且得到当前训练好以后鉴别器的损失
        d_loss_real = discriminator.train_on_batch(imgs, valid)
        #使用generator产生的图片，训练鉴别器，让他结果逼近fake（0），并且得到当前训练好以后鉴别器的损失
        #两个train_on_batch，相当于模型在当前权重参数下对真值为0和1的图片进行处理，计算得到损失，下一个epoch更新权重。
        d_loss_fake = discriminator.train_on_batch(gen_imgs, fake)
        #模型综合loss。
        d_loss = 0.5 * np.add(d_loss_real, d_loss_fake)

        # --------------------------- #
        #  训练generator
        # --------------------------- #
        noise = np.random.normal(0, 1, (batch_size, latent_dim))
        g_loss = combined.train_on_batch(noise, valid)
        print( g_loss)
        print("%d [D loss: %f, acc.: %.2f%%] [G loss: %f]" % (epoch, d_loss[0], 100 * d_loss[1], g_loss[0]))

        if epoch % sample_interval == 0:
            sample_images(epoch)

In [24]:
if not os.path.exists("gan_mnist"):
        os.makedirs("gan_mnist")
 #训练周期30000个，每200个训练周期，对当前的训练generator做一下预测并绘制图形。每次训练的batcha是256个       
train(epochs=30000, batch_size=256, sample_interval=200)


8/8 [==============================] - 0s 3ms/step
[0.9796692132949829, 0.16796875]
0 [D loss: 0.307759, acc.: 89.84%] [G loss: 0.979669]
8/8 [==============================] - 0s 3ms/step
[1.0642164945602417, 0.0703125]
1 [D loss: 0.283461, acc.: 93.75%] [G loss: 1.064216]
8/8 [==============================] - 0s 3ms/step
[1.2215338945388794, 0.03515625]
2 [D loss: 0.254284, acc.: 96.48%] [G loss: 1.221534]
8/8 [==============================] - 0s 3ms/step
[1.3435921669006348, 0.0234375]
3 [D loss: 0.212931, acc.: 99.02%] [G loss: 1.343592]
8/8 [==============================] - 0s 3ms/step
[1.4574040174484253, 0.0]
4 [D loss: 0.188725, acc.: 99.41%] [G loss: 1.457404]
8/8 [==============================] - 0s 3ms/step
[1.6021802425384521, 0.00390625]
5 [D loss: 0.163210, acc.: 100.00%] [G loss: 1.602180]
8/8 [==============================] - 0s 3ms/step
[1.737790822982788, 0.0]
6 [D loss: 0.146818, acc.: 100.00%] [G loss: 1.737791]
8/8 [==============================] - 0s 3ms/ste

8/8 [==============================] - 0s 2ms/step
[3.8043699264526367, 0.0]
61 [D loss: 0.048074, acc.: 99.61%] [G loss: 3.804370]
8/8 [==============================] - 0s 3ms/step
[3.8216214179992676, 0.0]
62 [D loss: 0.056453, acc.: 99.02%] [G loss: 3.821621]
8/8 [==============================] - 0s 3ms/step
[3.606977939605713, 0.0]
63 [D loss: 0.057803, acc.: 99.41%] [G loss: 3.606978]
8/8 [==============================] - 0s 3ms/step
[3.8598318099975586, 0.00390625]
64 [D loss: 0.060558, acc.: 98.63%] [G loss: 3.859832]
8/8 [==============================] - 0s 3ms/step
[3.6433281898498535, 0.01171875]
65 [D loss: 0.088457, acc.: 98.05%] [G loss: 3.643328]
8/8 [==============================] - 0s 3ms/step
[3.8483448028564453, 0.01171875]
66 [D loss: 0.056236, acc.: 98.24%] [G loss: 3.848345]
8/8 [==============================] - 0s 3ms/step
[3.6475653648376465, 0.01953125]
67 [D loss: 0.102559, acc.: 98.05%] [G loss: 3.647565]
8/8 [==============================] - 0s 3ms/ste

8/8 [==============================] - 0s 3ms/step
[2.152259349822998, 0.2421875]
121 [D loss: 0.513717, acc.: 76.95%] [G loss: 2.152259]
8/8 [==============================] - 0s 3ms/step
[3.105722665786743, 0.03515625]
122 [D loss: 0.281771, acc.: 84.96%] [G loss: 3.105723]
8/8 [==============================] - 0s 3ms/step
[3.4595019817352295, 0.0]
123 [D loss: 0.130513, acc.: 97.07%] [G loss: 3.459502]
8/8 [==============================] - 0s 3ms/step
[1.8526560068130493, 0.3046875]
124 [D loss: 0.490612, acc.: 77.34%] [G loss: 1.852656]
8/8 [==============================] - 0s 3ms/step
[2.483492374420166, 0.125]
125 [D loss: 0.316527, acc.: 84.96%] [G loss: 2.483492]
8/8 [==============================] - 0s 3ms/step
[3.302828311920166, 0.0]
126 [D loss: 0.115330, acc.: 97.27%] [G loss: 3.302828]
8/8 [==============================] - 0s 3ms/step
[1.9553191661834717, 0.2265625]
127 [D loss: 0.435152, acc.: 81.84%] [G loss: 1.955319]
8/8 [==============================] - 0s 4ms/

8/8 [==============================] - 0s 3ms/step
[1.412672758102417, 0.1796875]
181 [D loss: 0.616740, acc.: 60.94%] [G loss: 1.412673]
8/8 [==============================] - 0s 3ms/step
[1.7766474485397339, 0.0625]
182 [D loss: 0.485038, acc.: 73.63%] [G loss: 1.776647]
8/8 [==============================] - 0s 3ms/step
[1.4738444089889526, 0.1328125]
183 [D loss: 0.637109, acc.: 59.57%] [G loss: 1.473844]
8/8 [==============================] - 0s 3ms/step
[1.6939808130264282, 0.11328125]
184 [D loss: 0.539661, acc.: 65.43%] [G loss: 1.693981]
8/8 [==============================] - 0s 3ms/step
[1.1183223724365234, 0.23046875]
185 [D loss: 0.729878, acc.: 49.80%] [G loss: 1.118322]
8/8 [==============================] - 0s 3ms/step
[1.9602046012878418, 0.046875]
186 [D loss: 0.464828, acc.: 73.63%] [G loss: 1.960205]
8/8 [==============================] - 0s 3ms/step
[0.7164442539215088, 0.578125]
187 [D loss: 0.918784, acc.: 33.79%] [G loss: 0.716444]
8/8 [==========================

8/8 [==============================] - 0s 3ms/step
[0.6020959615707397, 0.984375]
240 [D loss: 0.656577, acc.: 48.24%] [G loss: 0.602096]
8/8 [==============================] - 0s 3ms/step
[0.6010925769805908, 0.99609375]
241 [D loss: 0.658955, acc.: 48.24%] [G loss: 0.601093]
8/8 [==============================] - 0s 3ms/step
[0.6038801670074463, 0.984375]
242 [D loss: 0.648053, acc.: 49.22%] [G loss: 0.603880]
8/8 [==============================] - 0s 3ms/step
[0.6033324599266052, 1.0]
243 [D loss: 0.651894, acc.: 48.05%] [G loss: 0.603332]
8/8 [==============================] - 0s 3ms/step
[0.6086388826370239, 0.98046875]
244 [D loss: 0.652764, acc.: 48.44%] [G loss: 0.608639]
8/8 [==============================] - 0s 3ms/step
[0.6089054942131042, 0.98828125]
245 [D loss: 0.655929, acc.: 48.83%] [G loss: 0.608905]
8/8 [==============================] - 0s 3ms/step
[0.602676272392273, 0.99609375]
246 [D loss: 0.657108, acc.: 47.66%] [G loss: 0.602676]
8/8 [===========================

...

8/8 [==============================] - 0s 3ms/step
[1.176245927810669, 0.03125]
2087 [D loss: 0.503338, acc.: 78.71%] [G loss: 1.176246]
8/8 [==============================] - 0s 3ms/step
[1.199341058731079, 0.0390625]
2088 [D loss: 0.495494, acc.: 79.10%] [G loss: 1.199341]
8/8 [==============================] - 0s 5ms/step
[1.1953227519989014, 0.02734375]
2089 [D loss: 0.501107, acc.: 81.64%] [G loss: 1.195323]
8/8 [==============================] - 0s 3ms/step
[1.188077688217163, 0.0390625]
2090 [D loss: 0.499885, acc.: 79.88%] [G loss: 1.188078]
8/8 [==============================] - 0s 3ms/step
[1.184163212776184, 0.03125]
2091 [D loss: 0.510064, acc.: 81.05%] [G loss: 1.184163]
8/8 [==============================] - 0s 4ms/step
[1.1791887283325195, 0.02734375]
2092 [D loss: 0.538477, acc.: 73.83%] [G loss: 1.179189]
8/8 [==============================] - 0s 3ms/step
[1.1441712379455566, 0.05078125]
2093 [D loss: 0.547419, acc.: 75.78%] [G loss: 1.144171]
8/8 [====================

8/8 [==============================] - 0s 4ms/step
[1.2555522918701172, 0.03515625]
2146 [D loss: 0.502028, acc.: 79.69%] [G loss: 1.255552]
8/8 [==============================] - 0s 4ms/step
[1.1929851770401, 0.0390625]
2147 [D loss: 0.600651, acc.: 70.70%] [G loss: 1.192985]
8/8 [==============================] - 0s 3ms/step
[1.2339580059051514, 0.03125]
2148 [D loss: 0.546308, acc.: 73.24%] [G loss: 1.233958]
8/8 [==============================] - 0s 3ms/step
[1.262439250946045, 0.03125]
2149 [D loss: 0.530903, acc.: 76.95%] [G loss: 1.262439]
8/8 [==============================] - 0s 3ms/step
[1.235079050064087, 0.03125]
2150 [D loss: 0.547053, acc.: 76.56%] [G loss: 1.235079]
8/8 [==============================] - 0s 4ms/step
[1.2130427360534668, 0.015625]
2151 [D loss: 0.546244, acc.: 71.29%] [G loss: 1.213043]
8/8 [==============================] - 0s 3ms/step
[1.2502657175064087, 0.01953125]
2152 [D loss: 0.542743, acc.: 73.83%] [G loss: 1.250266]
8/8 [=========================

[1.2310312986373901, 0.0078125]
2204 [D loss: 0.567579, acc.: 70.31%] [G loss: 1.231031]
8/8 [==============================] - 0s 4ms/step
[1.200270175933838, 0.01953125]
2205 [D loss: 0.525803, acc.: 74.22%] [G loss: 1.200270]
8/8 [==============================] - 0s 3ms/step
[1.2107019424438477, 0.0234375]
2206 [D loss: 0.573844, acc.: 70.12%] [G loss: 1.210702]
8/8 [==============================] - 0s 3ms/step
[1.2149200439453125, 0.015625]
2207 [D loss: 0.542508, acc.: 74.02%] [G loss: 1.214920]
8/8 [==============================] - 0s 4ms/step
[1.1517466306686401, 0.01953125]
2208 [D loss: 0.539607, acc.: 75.78%] [G loss: 1.151747]
8/8 [==============================] - 0s 3ms/step
[1.1659605503082275, 0.046875]
2209 [D loss: 0.517508, acc.: 75.78%] [G loss: 1.165961]
8/8 [==============================] - 0s 3ms/step
[1.1688542366027832, 0.05078125]
2210 [D loss: 0.521213, acc.: 76.37%] [G loss: 1.168854]
8/8 [==============================] - 0s 3ms/step
[1.2311075925827026,

8/8 [==============================] - 0s 3ms/step
[1.1231794357299805, 0.109375]
2263 [D loss: 0.573469, acc.: 69.14%] [G loss: 1.123179]
8/8 [==============================] - 0s 3ms/step
[1.1623176336288452, 0.08984375]
2264 [D loss: 0.526650, acc.: 75.59%] [G loss: 1.162318]
8/8 [==============================] - 0s 5ms/step
[1.1846387386322021, 0.0546875]
2265 [D loss: 0.555684, acc.: 73.05%] [G loss: 1.184639]
8/8 [==============================] - 0s 3ms/step
[1.1516908407211304, 0.0625]
2266 [D loss: 0.562878, acc.: 73.05%] [G loss: 1.151691]
8/8 [==============================] - 0s 3ms/step
[1.1559909582138062, 0.0546875]
2267 [D loss: 0.570071, acc.: 70.31%] [G loss: 1.155991]
8/8 [==============================] - 0s 3ms/step
[1.148809790611267, 0.0859375]
2268 [D loss: 0.577806, acc.: 69.53%] [G loss: 1.148810]
8/8 [==============================] - 0s 3ms/step
[1.1805975437164307, 0.046875]
2269 [D loss: 0.538495, acc.: 72.66%] [G loss: 1.180598]
8/8 [====================

KeyboardInterrupt: 